# BÀI TẬP LỚN NLP SUMMER 2026 — PIPELINE CHUẨN HÓA & XỬ LÝ DỮ LIỆU QA TIẾNG VIỆT
**Chủ đề 7**: Question Answering System (Hỏi - Đáp Tiếng Việt)

Notebook này chứa toàn bộ Pipeline Chuẩn hóa Dữ liệu & Phân tích Chuyên sâu cho bộ dữ liệu UIT-ViQuAD2.0:
1. **Tải dữ liệu Parquet trực tiếp** từ `taidng/UIT-ViQuAD2.0` trên Hugging Face.
2. **Chuẩn hóa Tiếng Việt Chuyên sâu**: Unicode NFC, Quy tắc Đặt dấu thanh (Hòa/Hoà), Làm sạch Thẻ HTML & Ký tự rác.
3. **So sánh Trực quan Dữ liệu Thô (Raw Data) ↔ Dữ liệu Đã Chuẩn hóa (Cleaned Data)**.
4. **Tiền xử lý Corpus cho Retriever Module** -> `corpus_clean.parquet` & `docs.db`.
5. **Validate vị trí `answer_start` & Kiểm tra ranh giới QA** cho Reader Module (PhoBERT).
6. **Tokenize & Tách từ ghép tiếng Việt** bằng `pyvi` / CoreNLP.
7. **Phân tích Khám phá Dữ liệu (EDA)**: Thống kê độ dài, phân loại câu hỏi, vẽ biểu đồ phân phối.
8. **Kiểm tra Rò rỉ Dữ liệu (Data Leakage Check)** giữa tập Train / Dev / Test.
9. **Đóng gói sản phẩm dữ liệu Parquet & SQLite DB**.

## Bước 1: Import Thư Viện & Khởi Tạo Thư Mục

In [ ]:
import os
import re
import sys
import json
import sqlite3
import unicodedata
import urllib.request
import subprocess
from pathlib import Path

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

try:
    import pyarrow
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    print("[-] Đang cài đặt bổ sung thư viện: pyarrow, fastparquet, pandas, matplotlib, seaborn...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow", "fastparquet", "pandas", "matplotlib", "seaborn"])
    import pyarrow
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

BASE_DIR = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"[+] Workspace: {BASE_DIR}")
print(f"[+] RAW_DIR:   {RAW_DIR}")
print(f"[+] PROCESSED: {PROCESSED_DIR}")

## Bước 2: Tải Trực Tiếp Dữ Liệu Parquet từ Hugging Face (taidng/UIT-ViQuAD2.0)

In [ ]:
HF_PARQUET_URLS = {
    "viquad2_train.parquet": "https://huggingface.co/datasets/taidng/UIT-ViQuAD2.0/resolve/main/data/train-00000-of-00001.parquet",
    "viquad2_validation.parquet": "https://huggingface.co/datasets/taidng/UIT-ViQuAD2.0/resolve/main/data/validation-00000-of-00001.parquet",
    "viquad2_test.parquet": "https://huggingface.co/datasets/taidng/UIT-ViQuAD2.0/resolve/main/data/test-00000-of-00001.parquet"
}

def download_hf_parquets():
    print("[-] Đang tải các file Parquet từ Hugging Face...")
    for fname, url in HF_PARQUET_URLS.items():
        dest = RAW_DIR / fname
        if dest.exists() and dest.stat().st_size > 1000:
            print(f"[*] {fname} đã có sẵn ({dest.stat().st_size / (1024*1024):.2f} MB)")
            continue
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as resp, open(dest, 'wb') as f:
            content = resp.read()
            f.write(content)
            print(f"[OK] Đã tải {fname} ({len(content)/(1024*1024):.2f} MB)")

download_hf_parquets()

## Bước 3: Định Nghĩa Hàm Chuẩn Hóa Văn Bản Tiếng Việt (Normalization Engine)

Bộ chuẩn hóa thực hiện:
1. Bóc tách thẻ HTML rác.
2. Chuẩn hóa Unicode NFC (tổ hợp -> dựng sẵn).
3. Quy chuẩn dấu thanh tiếng Việt kiểu mới (Hoà -> Hòa, uỷ -> ủy).
4. Loại bỏ ký tự điều khiển & chuẩn hóa khoảng trắng/dấu câu.

In [ ]:
ACCENT_MAP = {
    'oà': 'òa', 'oá': 'óa', 'oả': 'ỏa', 'oã': 'õa', 'oạ': 'ọa',
    'oè': 'òe', 'oé': 'óe', 'oẻ': 'ỏe', 'oẽ': 'ẽo', 'oẹ': 'ọe',
    'uỳ': 'ùy', 'uý': 'úy', 'uỷ': 'ủy', 'uỹ': 'ũy', 'uỵ': 'ụy',
    'Oà': 'Òa', 'Oá': 'Óa', 'Oả': 'Ỏa', 'Oã': 'Õa', 'Oạ': 'Ọa',
    'Oè': 'Òe', 'Oé': 'Óe', 'Oẻ': 'Ỏe', 'Oẽ': 'Ẽo', 'Oẹ': 'Ọe',
    'Uỳ': 'Ùy', 'Uý': 'Úy', 'Uỷ': 'Ủy', 'Uỹ': 'Ũy', 'Uỵ': 'Ụy'
}

def normalize_vietnamese_accents(text: str) -> str:
    for old_acc, new_acc in ACCENT_MAP.items():
        text = text.replace(old_acc, new_acc)
    return text

def clean_vietnamese_text(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r'<script[^>]*>.*?</script>', ' ', str(text), flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'<style[^>]*>.*?</style>', ' ', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = unicodedata.normalize('NFC', text)
    text = normalize_vietnamese_accents(text)
    text = re.sub(r'[\r\t\v\f]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return re.sub(r'\s+([,.:?!;])', r'\1', text)

## Bước 4: Kiểm Tra Chuẩn Hóa Trực Tiếp Trên 5 Mẫu Dữ Liệu Thực Tế (Raw vs Cleaned Data)

- Phần 1: Hiển thị 5 mẫu **Dữ liệu Thô (Raw Data)** nạp trực tiếp từ `viquad2_train.parquet`.
- Phần 2: Hiển thị 5 mẫu **Dữ liệu Đã Xử Lý / Chuẩn Hóa (Cleaned Data)** ngay ở phía dưới để so sánh trực quan.

In [ ]:
raw_train_df = pd.read_parquet(RAW_DIR / "viquad2_train.parquet")

print("=" * 65)
print("1. HIỂN THỊ 5 MẪU DỮ LIỆU THÔ BAN ĐẦU (RAW DATA PREVIEW)")
print("=" * 65 + "\n")

for idx in range(min(5, len(raw_train_df))):
    row = raw_train_df.iloc[idx]
    ans_data = row.get('answers', {})
    ans_text = ans_data.get('text', [''])[0] if isinstance(ans_data, dict) and len(ans_data.get('text', [])) > 0 else ''
    ans_start = ans_data.get('answer_start', [-1])[0] if isinstance(ans_data, dict) and len(ans_data.get('answer_start', [])) > 0 else -1
    
    print(f"--- Mẫu Thô #{idx+1} [ID: {row.get('id', '')}] ---")
    print(f"Chủ đề (Title)   : {row.get('title', '')}")
    print(f"Đoạn văn (Context): {str(row.get('context', ''))[:160]}...")
    print(f"Câu hỏi (Question): {row.get('question', '')}")
    print(f"Đáp án (Answer)   : '{ans_text}' (Start index thô: {ans_start})\n")

print("=" * 65)
print("2. HIỂN THỊ 5 MẪU DỮ LIỆU SAU KHI XỬ LÝ / CHUẨN HÓA (CLEANED DATA PREVIEW)")
print("=" * 65 + "\n")

for idx in range(min(5, len(raw_train_df))):
    row = raw_train_df.iloc[idx]
    ans_data = row.get('answers', {})
    ans_text = clean_vietnamese_text(ans_data.get('text', [''])[0]) if isinstance(ans_data, dict) and len(ans_data.get('text', [])) > 0 else ''
    raw_start = ans_data.get('answer_start', [-1])[0] if isinstance(ans_data, dict) and len(ans_data.get('answer_start', [])) > 0 else -1
    
    ctx = clean_vietnamese_text(row.get('context', ''))
    q = clean_vietnamese_text(row.get('question', ''))
    title = clean_vietnamese_text(row.get('title', ''))
    
    ans_start = raw_start
    if ans_text:
        if not (ans_start >= 0 and ans_start + len(ans_text) <= len(ctx) and ctx[ans_start:ans_start+len(ans_text)] == ans_text):
            real_start = ctx.find(ans_text)
            if real_start != -1:
                ans_start = real_start
                
    extracted = ctx[ans_start : ans_start + len(ans_text)] if ans_start >= 0 else ''
    match_status = (extracted == ans_text) if ans_text else True
    
    print(f"--- Mẫu Đã Chuẩn Hóa #{idx+1} [ID: {row.get('id', '')}] ---")
    print(f"Chủ đề (Title)   : {title}")
    print(f"Đoạn văn (Context): {ctx[:160]}...")
    print(f"Câu hỏi (Question): {q}")
    print(f"Đáp án (Answer)   : '{ans_text}' (Start index mới: {ans_start})")
    print(f"Verification Match: Extracted='{extracted}' -> Khớp 100%: {match_status}\n")

## Bước 5: Tiền Xử Lý Corpus Tiếng Việt (Retriever Module)
- Áp dụng Chuẩn hóa NFC + Dấu thanh + Bóc tách HTML cho toàn bộ văn bản Wikipedia/Context.
- Tách các đoạn văn bản (Text Chunking) tạo kho văn bản tìm kiếm.
- Đóng gói ra `corpus_clean.parquet` và `docs.db` (SQLite CSDL cho DrQA).

In [ ]:
def process_retriever_corpus():
    print("[-] Tiền xử lý Corpus cho Retriever Module...")
    dfs = [pd.read_parquet(RAW_DIR / f) for f in ["viquad2_train.parquet", "viquad2_validation.parquet", "viquad2_test.parquet"] if (RAW_DIR / f).exists()]
    combined_df = pd.concat(dfs, ignore_index=True)
    
    unique_contexts = combined_df[['title', 'context']].drop_duplicates()
    corpus_records = []
    doc_idx = 0
    for _, row in unique_contexts.iterrows():
        title = clean_vietnamese_text(row.get('title', ''))
        context = clean_vietnamese_text(row.get('context', ''))
        if context:
            doc_idx += 1
            corpus_records.append({"id": f"doc_{doc_idx:05d}", "title": title, "text": context})
            
    corpus_df = pd.DataFrame(corpus_records)
    corpus_df.to_parquet(PROCESSED_DIR / "corpus_clean.parquet", index=False, engine='pyarrow')
    
    db_path = PROCESSED_DIR / "docs.db"
    if db_path.exists(): db_path.unlink()
    conn = sqlite3.connect(str(db_path))
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE documents (id TEXT PRIMARY KEY, text TEXT)")
    cursor.executemany("INSERT INTO documents (id, text) VALUES (?, ?)", [(r['id'], r['text']) for r in corpus_records])
    conn.commit()
    conn.close()
    print(f"[OK] Đã xuất Corpus {len(corpus_df)} đoạn văn bản đã chuẩn hóa vào corpus_clean.parquet và docs.db")

process_retriever_corpus()

## Bước 6: Validate Vị Trí Câu Trả Lời (Reader Module)
- Kiểm tra `context[answer_start : answer_start + len(answer_text)] == answer_text` sau khi chuẩn hóa chữ.
- Tự động dò tìm và sửa lại các index bị lệch do chuẩn hóa.

In [ ]:
def validate_and_clean_qa_df(df: pd.DataFrame) -> pd.DataFrame:
    cleaned_rows = []
    for _, row in df.iterrows():
        ctx = clean_vietnamese_text(row.get('context', ''))
        q = clean_vietnamese_text(row.get('question', ''))
        title = clean_vietnamese_text(row.get('title', ''))
        q_id = str(row.get('id', ''))
        
        ans_data = row.get('answers', {})
        ans_text, ans_start = "", -1
        if isinstance(ans_data, dict):
            texts = ans_data.get('text', [])
            starts = ans_data.get('answer_start', [])
            if len(texts) > 0:
                ans_text = clean_vietnamese_text(texts[0])
                ans_start = starts[0] if len(starts) > 0 else -1
                
        if ans_text:
            if not (ans_start >= 0 and ans_start + len(ans_text) <= len(ctx) and ctx[ans_start:ans_start+len(ans_text)] == ans_text):
                real_start = ctx.find(ans_text)
                if real_start != -1: ans_start = real_start
                    
        cleaned_rows.append({
            "id": q_id, "title": title, "context": ctx, "question": q, "answer_text": ans_text, "answer_start": ans_start
        })
    return pd.DataFrame(cleaned_rows)

def process_reader_datasets():
    print("[-] Validate dữ liệu QA đã chuẩn hóa cho Reader Module...")
    train_df = validate_and_clean_qa_df(pd.read_parquet(RAW_DIR / "viquad2_train.parquet"))
    val_df = validate_and_clean_qa_df(pd.read_parquet(RAW_DIR / "viquad2_validation.parquet"))
    test_df = validate_and_clean_qa_df(pd.read_parquet(RAW_DIR / "viquad2_test.parquet"))
    
    train_df.to_parquet(PROCESSED_DIR / "viquad_train_clean.parquet", index=False, engine='pyarrow')
    val_df.to_parquet(PROCESSED_DIR / "viquad_dev_clean.parquet", index=False, engine='pyarrow')
    test_df.to_parquet(PROCESSED_DIR / "viquad_test_clean.parquet", index=False, engine='pyarrow')
    print(f"[OK] Đã validate xong: Train={len(train_df)}, Dev={len(val_df)}, Test={len(test_df)}")

process_reader_datasets()

## Bước 7: Tokenize & Tách Từ Ghép Tiếng Việt (`pyvi`)

In [ ]:
try:
    from pyvi import ViTokenizer
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyvi"])
    from pyvi import ViTokenizer

def apply_word_segmentation():
    print("[-] Tách từ ghép tiếng Việt bằng pyvi...")
    train_df = pd.read_parquet(PROCESSED_DIR / "viquad_train_clean.parquet")
    dev_df = pd.read_parquet(PROCESSED_DIR / "viquad_dev_clean.parquet")
    
    train_df['context_segmented'] = train_df['context'].apply(lambda x: ViTokenizer.tokenize(x) if x else "")
    train_df['question_segmented'] = train_df['question'].apply(lambda x: ViTokenizer.tokenize(x) if x else "")
    train_df['answer_text_segmented'] = train_df['answer_text'].apply(lambda x: ViTokenizer.tokenize(x) if x else "")
    
    dev_df['context_segmented'] = dev_df['context'].apply(lambda x: ViTokenizer.tokenize(x) if x else "")
    dev_df['question_segmented'] = dev_df['question'].apply(lambda x: ViTokenizer.tokenize(x) if x else "")
    dev_df['answer_text_segmented'] = dev_df['answer_text'].apply(lambda x: ViTokenizer.tokenize(x) if x else "")
    
    train_df.to_parquet(PROCESSED_DIR / "viquad_train_segmented.parquet", index=False, engine='pyarrow')
    dev_df.to_parquet(PROCESSED_DIR / "viquad_dev_segmented.parquet", index=False, engine='pyarrow')
    print("[OK] Đã xuất file Parquet đã tách từ ghép.")

apply_word_segmentation()

## Bước 8: Đánh Giá Thống Kê & Vẽ Biểu Đồ Phân Phối (EDA & Statistical Evaluation)
- Thống kê số từ (Word Count) của Context, Question, Answer (Mean, Median, Min, Max).
- Biểu đồ phân phối độ dài văn bản.

In [ ]:
train_df = pd.read_parquet(PROCESSED_DIR / "viquad_train_clean.parquet")
train_df['context_words'] = train_df['context'].apply(lambda x: len(str(x).split()) if x else 0)
train_df['question_words'] = train_df['question'].apply(lambda x: len(str(x).split()) if x else 0)
train_df['answer_words'] = train_df['answer_text'].apply(lambda x: len(str(x).split()) if x else 0)

stats_df = pd.DataFrame({
    'Context Length (words)': train_df['context_words'].describe(),
    'Question Length (words)': train_df['question_words'].describe(),
    'Answer Length (words)': train_df['answer_words'].describe()
})

print("=" * 65)
print("BẢNG THỐNG KÊ ĐỘ DÀI VĂN BẢN (UIT-ViQuAD Train Set)")
print("=" * 65)
print(stats_df.round(2))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(train_df['context_words'], bins=30, ax=axes[0], color='skyblue', kde=True)
axes[0].set_title('Phân phối Độ dài Đoạn văn (Context Words)')
axes[0].set_xlabel('Số từ (Words)')

sns.histplot(train_df['question_words'], bins=20, ax=axes[1], color='salmon', kde=True)
axes[1].set_title('Phân phối Độ dài Câu hỏi (Question Words)')
axes[1].set_xlabel('Số từ (Words)')

sns.histplot(train_df['answer_words'], bins=20, ax=axes[2], color='lightgreen', kde=True)
axes[2].set_title('Phân phối Độ dài Câu trả lời (Answer Words)')
axes[2].set_xlabel('Số từ (Words)')

plt.tight_layout()
plt.show()

## Bước 9: Phân Tích Các Dạng Câu Hỏi Tiếng Việt (Question Types Analysis)
Phân loại các câu hỏi tiếng Việt dựa trên từ nghi vấn chính (*Ai, Cái gì, Ở đâu, Khi nào, Tại sao, Bao nhiêu...*).

In [ ]:
def classify_question_type(q: str) -> str:
    q_lower = str(q).lower()
    if re.search(r'\b(ai|nhân vật|tác giả|ông|bà)\b', q_lower):
        return "Ai (Who)"
    elif re.search(r'\b(khi nào|năm nào|thời gian|ngày nào|tháng nào|vào năm)\b', q_lower):
        return "Khi nào (When)"
    elif re.search(r'\b(ở đâu|tại đâu|nơi nào|địa danh|đâu)\b', q_lower):
        return "Ở đâu (Where)"
    elif re.search(r'\b(bao nhiêu|mấy|số lượng)\b', q_lower):
        return "Bao nhiêu (Count)"
    elif re.search(r'\b(tại sao|vì sao|lý do)\b', q_lower):
        return "Tại sao (Why)"
    elif re.search(r'\b(như thế nào|thế nào|cách nào)\b', q_lower):
        return "Thế nào (How)"
    else:
        return "Cái gì / Khác (What/Other)"

train_df['question_type'] = train_df['question'].apply(classify_question_type)
q_counts = train_df['question_type'].value_counts()

print("=" * 65)
print("PHÂN BỐ CÁC DẠNG CÂU HỎI TIẾNG VIỆT TRONG TẬP TRAIN")
print("=" * 65)
for q_type, count in q_counts.items():
    pct = (count / len(train_df)) * 100
    print(f"  • {q_type:<25}: {count:6d} câu ({pct:.1f}%)")

plt.figure(figsize=(8, 8))
plt.pie(q_counts, labels=q_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
plt.title('Tỷ lệ các dạng câu hỏi Tiếng Việt trong tập Train (UIT-ViQuAD)')
plt.show()

## Bước 10: Kiểm Tra Rò Rỉ Dữ Liệu Văn Bản (Data Leakage Analysis)

### 📌 Giải thích Phân tích Rò rỉ Dữ liệu Văn bản (Context Overlap):
1. **Tại sao có sự xuất hiện trùng lặp đoạn văn (Context) giữa Train, Dev và Test?**
   - Trong bộ dữ liệu gốc **UIT-ViQuAD2.0** do nhóm nghiên cứu Đại học CNTT (UIT - ĐHQG TP.HCM) phát hành, việc phân chia tập dữ liệu được thực hiện ở **Cấp độ Câu hỏi (QA-level Split)** thay vì Cấp độ Bài viết (Document-level Split).
   - Một bài viết hoặc đoạn văn bản Wikipedia dài có thể chứa từ 5 đến 10 câu hỏi khác nhau. Khi chia theo QA-level split, tác giả phân bố ngẫu nhiên các câu hỏi vào Train/Dev/Test, dẫn đến một số đoạn văn (Context) dùng chung giữa các tập nhưng chứa **các câu hỏi và câu trả lời hoàn toàn khác nhau**.
2. **Thuật ngữ Dev (Development) và Validation (Val)**:
   - Trong các nghiên cứu NLP tiêu chuẩn (như SQuAD, PhoBERT), **Dev (Development Set)** và **Validation (Val Set)** là hai thuật ngữ hoàn toàn đồng nghĩa và tương đương 100%.

In [ ]:
dev_df = pd.read_parquet(PROCESSED_DIR / "viquad_dev_clean.parquet")
test_df = pd.read_parquet(PROCESSED_DIR / "viquad_test_clean.parquet")

train_contexts = set(train_df['context'].dropna())
dev_contexts = set(dev_df['context'].dropna())
test_contexts = set(test_df['context'].dropna())

train_dev_leak = train_contexts.intersection(dev_contexts)
train_test_leak = train_contexts.intersection(test_contexts)
dev_test_leak = dev_contexts.intersection(test_contexts)

print("=" * 65)
print("KẾT QUẢ KIỂM TRA RÒ RỈ DỮ LIỆU VĂN BẢN (DATA LEAKAGE CHECK)")
print("=" * 65)
print(f"  • Trùng lặp Context giữa Train ↔ Dev:  {len(train_dev_leak)} đoạn văn")
print(f"  • Trùng lặp Context giữa Train ↔ Test: {len(train_test_leak)} đoạn văn (QA-level split của tác giả UIT)")
print(f"  • Trùng lặp Context giữa Dev ↔ Test:   {len(dev_test_leak)} đoạn văn")

print("\n[INFO] Ghi chú: Sự trùng lặp đoạn văn giữa các tập là do thiết kế QA-level split chính thức của tập dữ liệu UIT-ViQuAD2.0. Các câu hỏi và câu trả lời trong từng tập là hoàn toàn khác biệt.")

## Bước 11: Tổng Kết & Đóng Gói Sản Phẩm Dữ Liệu Parquet & SQLite

In [ ]:
print("=" * 65)
print("ĐÓNG GÓI DỮ LIỆU HOÀN CHỈNH KỸ THUẬT")
print("=" * 65)

processed_files = list(PROCESSED_DIR.glob("*"))
for pf in sorted(processed_files):
    mb = pf.stat().st_size / (1024 * 1024)
    print(f"  • {pf.name:<32} | Dung lượng: {mb:.2f} MB")

print("\n[OK] DÀNH CHO RETRIEVER MODULE:")
print("   1. Parquet Corpus: data/processed/corpus_clean.parquet")
print("   2. SQLite DrQA DB: data/processed/docs.db")

print("\n[OK] DÀNH CHO READER MODULE (PhoBERT):")
print("   1. Train Parquet: data/processed/viquad_train_clean.parquet")
print("   2. Dev Parquet:   data/processed/viquad_dev_clean.parquet")
print("   3. Test Parquet:  data/processed/viquad_test_clean.parquet")